# 30문항 BM25 단독·RRF·재정렬 검색 비교

통합 노트북에서 검색 단계 비교를 분리했습니다. 답변·채점 API 호출 없이 두 BM25 방식의 상위 20개를 기록합니다.

실행 출력은 GitHub 공유를 위해 제거했습니다. 원본 문서와 평가 데이터는 별도로 준비해야 합니다.


In [ ]:
%pip install -q numpy pandas faiss-cpu "sentence-transformers==5.6.0" langchain-huggingface langchain-community langchain-text-splitters pymupdf4llm langchain-hwp-hwpx-loader "transformers==4.51.3" accelerate rank_bm25 kiwipiepy


In [ ]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm
from google.colab import drive
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/data"
metadata_csv = "/content/drive/MyDrive/data_list.csv"


In [ ]:
# 문서 순서를 고정해 다시 실행해도 청크 번호가 같은 순서로 생성되도록 합니다.
sorted_file_names = sorted(
    os.listdir(folder),
    key=lambda name: (unicodedata.normalize("NFKC", name).casefold(), name),
)
print("정렬한 파일 수:", len(sorted_file_names))

## 1. 문서 로드 (HWP, PDF)


In [ ]:
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()

df = pd.read_csv(metadata_csv, encoding="utf-8-sig")
loaded_files = []
for name in sorted_file_names:
    full_path = os.path.join(folder, name)
    if not os.path.isfile(full_path):
        continue
    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp", "hwpx"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = "\n\n".join(doc.page_content for doc in loader.load())
    else:
        continue
    if not text.strip():
        raise ValueError(f"본문이 비어 있습니다: {name}")
    loaded_files.append((name, text))
if not loaded_files:
    raise ValueError(f"PDF/HWP/HWPX 문서를 찾지 못했습니다: {folder}")
print("로드한 문서:", len(loaded_files))


## 2. 메타데이터와 본문을 Document로 저장

`page_content`에 본문을, `metadata`에 CSV의 문서 정보를 저장합니다.


In [ ]:
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]
missing_columns = set(metadata_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"data_list.csv에 없는 열: {sorted(missing_columns)}")
documents = []
unmatched_files = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        unmatched_files.append(name)
        continue
    if len(result_df[["발주 기관", "사업명"]].drop_duplicates()) != 1:
        raise ValueError(f"파일의 사업 메타데이터가 여러 개입니다: {name}")
    row = result_df.iloc[0]
    metadata = {column: "" if pd.isna(row[column]) else str(row[column])
                for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))
if not documents:
    raise ValueError("문서 파일명과 data_list.csv의 파일명을 확인하세요.")
print("메타데이터와 연결한 문서:", len(documents))
if unmatched_files:
    print("CSV와 연결되지 않아 제외한 파일:", unmatched_files)


## 3. 본문 기본 전처리

연속 공백과 빈 줄만 정리합니다.


In [ ]:
for document in documents:
    text = document.page_content.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    document.page_content = re.sub(r"\n{3,}", "\n\n", text).strip()


## 4. 청킹


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))


## 5. 임베딩 모델 준비

사업별 FAISS 인덱스는 해당 사업의 질문이 처음 들어올 때 만듭니다.


In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

model_device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "dragonkue/BGE-m3-ko"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": model_device, "trust_remote_code": True},
    encode_kwargs={"batch_size": 16, "normalize_embeddings": True},
)
print("임베딩 모델:", embedding_model_name, "/ 장치:", model_device)


## 6. 질문에서 사업 식별 후 하이브리드 검색

질문에 적힌 정식 사업명을 문서 메타데이터의 사업명 목록과 비교합니다. 띄어쓰기와 문장부호 차이는 무시합니다. 줄임말은 `PROJECT_ALIASES`에 등록하세요. 사업을 하나로 특정할 수 없으면 오류를 내며, 평가 데이터의 `project` 필드는 검색에 사용하지 않습니다.


In [ ]:
from collections import defaultdict
from langchain_community.retrievers import BM25Retriever


def normalize_project_text(value):
    return "".join(
        char for char in unicodedata.normalize("NFKC", str(value)).casefold()
        if char.isalnum()
    )


def project_title_matches_query(project_key, normalized_query):
    agency, title = project_key
    normalized_title = normalize_project_text(title)
    if normalized_title in normalized_query:
        return True

    # 기관명이 사업명 앞에 있지만 질문에서는 조사와 함께 분리된 경우
    normalized_agency = normalize_project_text(agency)
    if normalized_agency and normalized_title.startswith(normalized_agency):
        short_title = normalized_title[len(normalized_agency):]
        if len(short_title) >= 8 and normalized_agency in normalized_query:
            return short_title in normalized_query
    return False


def project_identity_matches(project_key, expected_project):
    agency, title = project_key
    expected = normalize_project_text(expected_project)
    return expected in {
        normalize_project_text(title),
        normalize_project_text(agency + title),
    }


def metadata_text(value):
    return "" if pd.isna(value) else str(value).strip()


# 줄임말을 쓰려면 아래에 "줄임말": ("발주 기관", "정식 사업명")을 추가합니다.
PROJECT_ALIASES = {}

project_chunks_by_key = defaultdict(list)
for doc in chunks:
    agency = metadata_text(doc.metadata.get("발주 기관"))
    title = metadata_text(doc.metadata.get("사업명"))
    if title:
        project_chunks_by_key[(agency, title)].append(doc)

project_catalog = sorted(project_chunks_by_key)
if not project_catalog:
    raise ValueError("사업명 메타데이터가 있는 청크가 없습니다.")
print("식별 가능한 사업:", len(project_catalog))


def choose_project(matches, query):
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]

    normalized_query = normalize_project_text(query)
    agency_matches = [
        key for key in matches
        if key[0] and normalize_project_text(key[0]) in normalized_query
    ]
    if len(agency_matches) == 1:
        return agency_matches[0]

    labels = ", ".join(f'[{agency}] "{title}"' for agency, title in matches)
    raise ValueError(f"사업을 하나로 특정할 수 없습니다: {labels}")


def identify_project(query):
    """질문의 정식 사업명 또는 등록된 줄임말로 사업 하나를 고릅니다."""
    normalized_query = normalize_project_text(query)
    if not normalized_query:
        raise ValueError("질문이 비어 있습니다.")

    # 정식 사업명이 질문에 들어 있으면 포함 관계의 짧은 이름만 제외합니다.
    direct_matches = [
      key for key in project_catalog
      if project_title_matches_query(key, normalized_query)
    ]
    if direct_matches:
        specific_matches = [
            key for key in direct_matches
            if not any(
                normalize_project_text(key[1]) != normalize_project_text(other[1])
                and normalize_project_text(key[1]) in normalize_project_text(other[1])
                for other in direct_matches
            )
        ]
        return choose_project(specific_matches, query)

    # 정식 명칭이 없는 질문은 사용자가 등록한 줄임말만 사용합니다.
    alias_matches = [
        (alias, target) for alias, target in PROJECT_ALIASES.items()
        if normalize_project_text(alias) in normalized_query
    ]
    if alias_matches:
        longest = max(len(normalize_project_text(alias)) for alias, _ in alias_matches)
        targets = [
            target for alias, target in alias_matches
            if len(normalize_project_text(alias)) == longest
        ]
        unknown = [target for target in targets if target not in project_chunks_by_key]
        if unknown:
            raise ValueError(f"PROJECT_ALIASES의 사업이 문서 목록에 없습니다: {unknown}")
        return choose_project(targets, query)

    raise ValueError(
        "질문에서 사업명을 찾지 못했습니다. 정식 사업명을 질문에 적거나 "
        "PROJECT_ALIASES에 줄임말을 등록해 주세요."
    )


from kiwipiepy import Kiwi

kiwi = Kiwi()
BM25_MODES = ("current", "kiwi")
BM25_MODE = "current"
PUNCTUATION_TAGS = {"SF", "SP", "SS", "SE", "SO", "SW"}


def kiwi_morphs(text):
    """문서와 질문에 같은 Kiwi 형태소 분석을 적용합니다."""
    return [
        token.form
        for token in kiwi.tokenize(text)
        if token.form.strip() and token.tag not in PUNCTUATION_TAGS
    ]


dense_retrievers_by_project = {}
bm25_retrievers_by_project_and_mode = {}


def get_retrievers(project_key, bm25_mode):
    """사업별 검색기를 필요할 때 한 번만 만들고 재사용합니다."""
    if bm25_mode not in BM25_MODES:
        raise ValueError(f"지원하지 않는 BM25 방식: {bm25_mode}")

    project_chunks = project_chunks_by_key[project_key]
    if project_key not in dense_retrievers_by_project:
        store = FAISS.from_documents(project_chunks, embeddings)
        dense_retrievers_by_project[project_key] = store.as_retriever(search_kwargs={"k": 20})

    bm25_key = (project_key, bm25_mode)
    if bm25_key not in bm25_retrievers_by_project_and_mode:
        options = {"preprocess_func": kiwi_morphs} if bm25_mode == "kiwi" else {}
        bm25 = BM25Retriever.from_documents(project_chunks, **options)
        bm25.k = 20
        bm25_retrievers_by_project_and_mode[bm25_key] = bm25

    return (
        dense_retrievers_by_project[project_key],
        bm25_retrievers_by_project_and_mode[bm25_key],
    )


def rrf_search(query, project_key, bm25_mode, k=60):
    """BGE-M3와 BM25의 검색 순위를 RRF 점수로 합칩니다."""
    dense_retriever, bm25_retriever = get_retrievers(project_key, bm25_mode)
    dense_results = dense_retriever.invoke(query)
    sparse_results = bm25_retriever.invoke(query)
    scores = defaultdict(float)
    documents_by_id = {}

    for results in (dense_results, sparse_results):
        for rank, doc in enumerate(results, start=1):
            doc_id = doc.metadata["chunk_id"]
            documents_by_id.setdefault(doc_id, doc)
            scores[doc_id] += 1 / (k + rank)

    ranked_ids = sorted(scores, key=scores.get, reverse=True)
    return [documents_by_id[doc_id] for doc_id in ranked_ids]


## 30문항 검색 단계 비교

기본·Kiwi BM25 단독, RRF 결합, 재순위화의 상위 20개 청크를 각각 기록합니다. 답변 생성 및 API 채점은 실행하지 않습니다.


In [ ]:
import hashlib
BM25_MODE = 'current'
def make_search_text(query, method):
    if method != 'basic':
        raise ValueError('이 검색 비교에서는 basic 방식만 사용합니다.')
    return query


## 8. 질문에서 식별한 사업 안에서 재순위화


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True,
    device=model_device,
)

# 질문과 검색 설정이 같을 때만 검색 결과를 재사용합니다.
retrieval_cache = {}


def retrieve_and_rerank(query, method, bm25_mode=None):
    """사업 범위에서 검색한 뒤 상위 문서를 다시 정렬합니다."""
    bm25_mode = BM25_MODE if bm25_mode is None else bm25_mode
    project_key = identify_project(query)
    cache_key = (project_key, method, query, bm25_mode)
    if cache_key not in retrieval_cache:
        search_text = make_search_text(query, method)
        found = rrf_search(search_text, project_key, bm25_mode)
        ranking = reranker.rank(
            query,
            [doc.page_content for doc in found],
            top_k=20,
            batch_size=8,
        )
        reranked_docs = [found[row["corpus_id"]] for row in ranking]
        retrieval_cache[cache_key] = (
            project_key, search_text, found, reranked_docs
        )
    return retrieval_cache[cache_key]


### 평가 질문 불러오기: 3개 사업 × 10문항

파일에 기록된 사업 순서를 유지하고 각 사업의 첫 10문항을 선택합니다.
기준 답변과 사업 정답은 채점에 사용합니다. 검색의 사업 식별에는 질문만 사용합니다.


In [ ]:
import json
from collections import Counter
from pathlib import Path

EVAL_PROJECT_COUNT = 3
EVAL_QUESTIONS_PER_PROJECT = 10
EXPECTED_QUESTION_COUNT = EVAL_PROJECT_COUNT * EVAL_QUESTIONS_PER_PROJECT
EVAL_LABEL = f"{EVAL_PROJECT_COUNT}x{EVAL_QUESTIONS_PER_PROJECT}"
eval_dataset_candidates = [
    Path("/content/drive/MyDrive/rag_eval_dataset_3x10.json"),
    Path("/content/drive/MyDrive/rag_eval_dataset.json"),
    Path("rag_eval_dataset_3x10.json"),
    Path("rag_eval_dataset.json"),
]
eval_dataset_path = next((p for p in eval_dataset_candidates if p.is_file()), None)
if eval_dataset_path is None:
    raise FileNotFoundError("평가 JSON을 Google Drive 루트 또는 노트북 폴더에 두세요.")
with eval_dataset_path.open(encoding="utf-8-sig") as file:
    all_eval_records = json.load(file)
if not isinstance(all_eval_records, list) or not all_eval_records:
    raise ValueError("평가 JSON은 문항 객체들의 비어 있지 않은 목록이어야 합니다.")
if any(not isinstance(r, dict) or not r.get("project") for r in all_eval_records):
    raise ValueError("사업명(project)이 없는 평가 문항이 있습니다.")
available_projects = list(dict.fromkeys(r["project"] for r in all_eval_records))
selected_projects = available_projects[:EVAL_PROJECT_COUNT]
selected_eval_records = []
selected_count = Counter()
for record in all_eval_records:
    project = record["project"]
    if project in selected_projects and selected_count[project] < EVAL_QUESTIONS_PER_PROJECT:
        selected_eval_records.append(record)
        selected_count[project] += 1
if len(selected_projects) != EVAL_PROJECT_COUNT or any(
    selected_count[p] != EVAL_QUESTIONS_PER_PROJECT for p in selected_projects
):
    raise ValueError(f"3개 사업에서 각 10문항을 선택하지 못했습니다: {dict(selected_count)}")
evaluation_data = []
for record in selected_eval_records:
    fields = ("id", "question", "reference", "project")
    if not all(isinstance(record.get(f), str) and record[f].strip() for f in fields):
        raise ValueError(f"필수 평가 필드가 비어 있거나 문자열이 아닙니다: {record.get('id')}")
    evaluation_data.append({
        "id": record["id"], "query": record["question"],
        "reference": record["reference"], "project": record["project"],
        "question_type": record.get("question_type", ""),
        "evidence_quotes": record.get("evidence_quotes", []),
    })
for field in ("id", "query"):
    if len({r[field] for r in evaluation_data}) != EXPECTED_QUESTION_COUNT:
        raise ValueError(f"문항 수가 맞지 않거나 {field}가 중복됩니다.")
print(f"평가셋: {eval_dataset_path} / {len(evaluation_data)}문항")
print("사업별:", dict(Counter(r["project"] for r in evaluation_data)))
print("유형별:", dict(Counter(r["question_type"] for r in evaluation_data)))


### 기본 BM25와 Kiwi BM25의 단독 검색 결과 저장
각 사업의 동일한 청크에서 두 검색 방식의 상위 20개를 뽑아 JSON으로 저장합니다.


In [ ]:
assert len(evaluation_data) == 30
assert len({r["id"] for r in evaluation_data}) == 30

project_keys = {
    r["id"]: identify_project(r["query"])
    for r in evaluation_data
}

corpus = {}
indexes = {}

for key in dict.fromkeys(project_keys.values()):
    docs = project_chunks_by_key[key]
    assert len(docs) >= 20

    for doc in docs:
        cid = str(doc.metadata["chunk_id"])
        assert cid not in corpus, f"중복 청크 ID: {cid}"
        corpus[cid] = {
            "text": doc.page_content,
            "agency": key[0],
            "project": key[1],
        }

    normal = BM25Retriever.from_documents(docs)
    kiwi_index = BM25Retriever.from_documents(
        docs, preprocess_func=kiwi_morphs
    )
    normal.k = kiwi_index.k = 20
    indexes[key] = {"current": normal, "kiwi": kiwi_index}

questions = []
for number, item in enumerate(evaluation_data, 1):
    key = project_keys[item["id"]]
    ranked = {
        mode: [
            str(doc.metadata["chunk_id"])
            for doc in index.invoke(item["query"])
        ]
        for mode, index in indexes[key].items()
    }
    questions.append({
        "id": item["id"],
        "question": item["query"],
        "project": item["project"],
        "agency": key[0],
        "corpus_project": key[1],
        "reference": item["reference"],
        "evidence_quotes": item.get("evidence_quotes", []),
        "ranked_chunk_ids": ranked,
    })
    print(f"[{number}/30] {item['id']} 검색 완료")

output = {
    "comparison_type": "bm25_only",
    "corpus_sha256": hashlib.sha256(
        json.dumps(
            corpus, ensure_ascii=False, sort_keys=True
        ).encode("utf-8")
    ).hexdigest(),
    "corpus": corpus,
    "questions": questions,
}

path = Path("/content/drive/MyDrive/bm25_only_comparison_30.json")
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("저장:", path)


### RRF 결합과 재순위화 결과 저장
두 BM25 방식의 RRF 후보와 재순위화된 상위 20개를 JSON으로 저장합니다.


In [ ]:
assert len(evaluation_data) == 30

records = []
corpus = {}

for number, item in enumerate(evaluation_data, 1):
    project_key = identify_project(item["query"])

    # 같은 원문을 사용했는지 확인할 수 있도록 함께 저장
    for doc in project_chunks_by_key[project_key]:
        corpus[str(doc.metadata["chunk_id"])] = {
            "text": doc.page_content,
            "agency": project_key[0],
            "project": project_key[1],
        }

    ranked = {}

    for mode in ("current", "kiwi"):
        _, _, candidates, final_docs = retrieve_and_rerank(
            item["query"],
            "basic",
            bm25_mode=mode,
        )

        final_ids = [
            str(doc.metadata["chunk_id"]) for doc in final_docs
        ]
        assert len(final_ids) == len(set(final_ids)) == 20

        ranked[mode] = {
            "rrf_top20": [
                str(doc.metadata["chunk_id"])
                for doc in candidates[:20]
            ],
            "reranked_top20": final_ids,
        }

    records.append({
        "id": item["id"],
        "question": item["query"],
        "reference": item["reference"],
        "project": item["project"],
        "agency": project_key[0],
        "corpus_project": project_key[1],
        "ranked_chunk_ids": ranked,
    })

    print(f"[{number}/30] {item['id']} 재정렬 20개 완료")

output = {
    "comparison_type": "hybrid_reranked_top20",
    "corpus_sha256": hashlib.sha256(
        json.dumps(
            corpus, ensure_ascii=False, sort_keys=True
        ).encode("utf-8")
    ).hexdigest(),
    "corpus": corpus,
    "questions": records,
}

output_dir = (
    Path(eval_dataset_path).parent
    if "eval_dataset_path" in globals()
    else Path("/content/drive/MyDrive")
)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "hybrid_reranked_top20_30.json"
output_path.write_text(
    json.dumps(output, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("저장 완료:", output_path)
